# 007 Stage 2: Agent→Target Allocation GNN Training

**Prerequisite**: 006 has already trained the Stage 1 (source→agent weight prediction) model.

This notebook implements the Stage 2 allocation system:
1. Load the frozen Stage 1 model → infer W_sa → compute agent_demand
2. Build the agent-target bipartite graph (AllocationGraphBuilder)
3. Train the Stage 2 allocation model (AllocationSolver)
4. Infer → W_at → allocate demand to targets
5. Evaluate (RMSE/Corr vs baseline)

In [ ]:
import sys
import pickle
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import torch
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler

from SpatialAllocation.GNN.core.ModelConfig import ModelConfig
from SpatialAllocation.GNN.core.EdgeWeightSolver import EdgeWeightSolver
from SpatialAllocation.GNN.Allocation import (
    AllocationConfig, AllocationSolver, build_allocation_graph
)

# ====== Path configuration (aligned with 006) ======
DATA_DIR = Path('./results/intermediate')
ASSEMBLED_DIR = DATA_DIR / 'features' / 'assembled'

# Stage 1 experiment name (model trained in 006)
STAGE1_EXPERIMENT = 'hgt_0_1'
STAGE1_GNN_DIR = DATA_DIR / 'GNN_TEST' / STAGE1_EXPERIMENT
STAGE1_MODEL_PATH = str(STAGE1_GNN_DIR / 'model_42_hgt_test.pth')

# Stage 2 experiment name and output directory
EXPERIMENT_NAME = 'stage2_distance_homogeneity'
ALLOC_DIR = DATA_DIR / 'Allocation_TEST' / EXPERIMENT_NAME
ALLOC_DIR.mkdir(parents=True, exist_ok=True)

# ====== Region configuration (consistent with 006) ======
TRAIN_LOCATIONS = [
    'London',
    'TLH2', 'TLH3', 'TLJ1', 'TLF1', 'TLF2',
    'TLC1', 'TLC2', 'TLD6', 'TLG1', 'TLG2', 'TLE4',
]
TEST_LOCATIONS = ['TLH1', 'TLE3', 'TLD3', 'TLD4']
STUDY_REGIONS = TRAIN_LOCATIONS + TEST_LOCATIONS

# k-NN parameters
K_NEAREST_TARGETS = 5

# ====== Agent feature list (controls which features enter the allocation graph) ======
# Modify this list to switch between different feature-combination experiments
AGENT_FEATURE_COLS = [
    # landuse proportions
    'lu_residential_prop', 'lu_commercial_prop', 'lu_industrial_prop',
    'lu_agricultural_prop', 'lu_others_prop',
    # world cover
    # 'wc_built_up_ratio', 'wc_agricultural_ratio', 'wc_others_ratio',
    # spectral indices
    # 'NDVI_mean', 'NDVI_std', 'NDVI_median',
    # 'NDBI_mean', 'NDBI_std', 'NDBI_median',
    # 'NDWI_mean', 'NDWI_std', 'NDWI_median',
    # 'BSI_mean', 'BSI_std', 'BSI_median',
    # 'UI_mean', 'UI_std', 'UI_median',
    # demand quantiles
    # 'q_residential', 'q_commercial', 'q_industrial',
    # 'q_agricultural', 'q_others',
]

# Whether to concatenate the Stage 1 predicted agent-weighted demand as an extra feature
USE_STAGE1_WEIGHT_FEATURE = False

print(f'Train regions: {len(TRAIN_LOCATIONS)}')
print(f'Test regions: {len(TEST_LOCATIONS)}')
print(f'Agent features: {len(AGENT_FEATURE_COLS)}-dim {AGENT_FEATURE_COLS}')
print(f'Stage 1 weight feature: {"enabled" if USE_STAGE1_WEIGHT_FEATURE else "disabled"}')
print(f'Stage 1 model: {STAGE1_MODEL_PATH}')
print(f'Stage 2 output: {ALLOC_DIR}')

In [ ]:
# ─── Load data (aligned with 006 Cell 2) ───
region_gdf = gpd.read_file(str(DATA_DIR / 'ITL3_region.gpkg'))
substations_gdf = gpd.read_file(str(DATA_DIR / 'substations.gpkg'))

print(f'ITL3 regions: {region_gdf.shape[0]} rows')
print(f'Substations: {substations_gdf.shape[0]} rows')

# Load per-region assembled grid_gdf (consistent with 006 Cell 2)
grids = {}
for loc in STUDY_REGIONS:
    path = ASSEMBLED_DIR / f'{loc}_grid_points.pickle'
    with open(path, 'rb') as f:
        grid_gdf, step_size_m = pickle.load(f)
    grids[loc] = (grid_gdf, step_size_m)
    print(f'  {loc}: {len(grid_gdf)} grid points, step={step_size_m}m')

# Organize data dicts by region (consistent with 006 Cell 2: matched via the ITL3 column)
region_dict = {}
subs_dict = {}

for loc in STUDY_REGIONS:
    grid_gdf, step_size_m = grids[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    region_dict[loc] = region_gdf[region_gdf['ITL3'].isin(study_itl3)].copy()
    subs_dict[loc] = substations_gdf[substations_gdf['ITL3'].isin(study_itl3)].copy().reset_index(drop=True)
    print(f'  {loc}: {len(region_dict[loc])} ITL3 regions, {len(subs_dict[loc])} substations')

print('\nData loading complete.')

In [ ]:
# ─── Load the Stage 1 model and run inference per region to obtain W_sa ───
from SpatialAllocation.GNN.utils.GraphBuilder import preprocess_features

# Load the pre-built Stage 1 graphs (already saved to STAGE1_GNN_DIR in 006)
stage1_graphs = {}
for loc in STUDY_REGIONS:
    graph_path = STAGE1_GNN_DIR / f'{loc}_graph.pt'
    if graph_path.exists():
        stage1_graphs[loc] = torch.load(graph_path, weights_only=False)
        print(f'{loc}: Stage 1 graph loaded')
    else:
        print(f'{loc}: graph file not found ({graph_path}); please run the 006 notebook first')

# Initialize the Stage 1 solver and load the model
stage1_config = ModelConfig(
    conv_type='hgt',
    allocation_temperature_start=0.1,
    hidden_dim=64,
    embedding_dim=32,
    num_layers=2,
    save_path=STAGE1_MODEL_PATH,
)

# Infer input dimensions from a sample graph and initialize the model structure
sample_loc = STUDY_REGIONS[0]
sample_dl = DataLoader([stage1_graphs[sample_loc]], batch_size=1)
stage1_solver = EdgeWeightSolver(stage1_config)
stage1_solver.init_model(sample_dl)

# Run inference for each region
agent_demands = {}  # loc -> (num_a,) ndarray
edge_weights_dfs = {}  # loc -> DataFrame

for loc in STUDY_REGIONS:
    if loc not in stage1_graphs:
        continue

    result_df = stage1_solver.predict_edge_weights(stage1_graphs[loc])
    edge_weights_dfs[loc] = result_df

    # Compute agent allocation demand: W_sa × source_demand
    graph = stage1_graphs[loc]
    source_demand = graph['source'].y.cpu().numpy()  # (num_s,)

    # Weighted demand per agent = ∑(w_sa × demand_s)
    num_a = graph['agent'].num_nodes
    agent_demand = np.zeros(num_a)
    for _, row in result_df.iterrows():
        s_idx = int(row['source_node_idx'])
        a_idx = int(row['agent_node_idx'])
        agent_demand[a_idx] += row['predicted_weight'] * source_demand[s_idx]

    agent_demands[loc] = agent_demand
    print(f'{loc}: agent_demand sum={agent_demand.sum():.1f} MVA '
          f'(source_demand sum={source_demand.sum():.1f} MVA)')

print('\nStage 1 inference complete.')

In [ ]:
# ─── Build the agent-target allocation graph ───
allocation_graphs = {}

for loc in STUDY_REGIONS:
    grid_gdf, step_size_m = grids[loc]
    subs_sub = subs_dict[loc]

    # Project to EPSG:3857 + normalize (consistent with 006)
    gdf_a = grid_gdf.copy().to_crs('EPSG:3857')
    gdf_t = subs_sub.copy().to_crs('EPSG:3857')

    coords_a = np.column_stack([gdf_a.geometry.x, gdf_a.geometry.y])
    coords_t = np.column_stack([gdf_t.geometry.x, gdf_t.geometry.y])
    scaler = StandardScaler().fit(np.vstack([coords_a, coords_t]))

    from shapely.geometry import Point
    coords_a_scaled = scaler.transform(coords_a)
    coords_t_scaled = scaler.transform(coords_t)

    gdf_a_scaled = gdf_a.copy()
    gdf_a_scaled['geometry'] = [Point(x, y) for x, y in coords_a_scaled]

    gdf_t_scaled = gdf_t.copy()
    gdf_t_scaled['geometry'] = [Point(x, y) for x, y in coords_t_scaled]

    # Add agent demand to the GeoDataFrame
    if loc in agent_demands:
        gdf_a_scaled['Demand (MVA)'] = agent_demands[loc]

    # Add target demand
    if 'Demand (MVA)' not in gdf_t_scaled.columns:
        warnings.warn(f'{loc}: gdf_t is missing the Demand (MVA) column')

    # Filter to the feature columns actually present in gdf_a
    agent_cols = [c for c in AGENT_FEATURE_COLS if c in gdf_a_scaled.columns]
    if len(agent_cols) < len(AGENT_FEATURE_COLS):
        missing = set(AGENT_FEATURE_COLS) - set(agent_cols)
        warnings.warn(f'{loc}: missing feature columns {missing}')

    alloc_graph = build_allocation_graph(
        gdf_agent=gdf_a_scaled,
        gdf_target=gdf_t_scaled,
        agent_feature_cols=agent_cols,
        agent_weights=agent_demands.get(loc) if USE_STAGE1_WEIGHT_FEATURE else None,
        k=K_NEAREST_TARGETS,
    )

    allocation_graphs[loc] = alloc_graph
    torch.save(alloc_graph, ALLOC_DIR / f'{loc}_alloc_graph.pt')

    split = 'TRAIN' if loc in TRAIN_LOCATIONS else 'TEST'
    print(f'{loc} [{split}]: agent={alloc_graph["agent"].x.shape}, '
          f'target={alloc_graph["target"].x.shape}, '
          f'edges={alloc_graph["agent", "connects_to", "target"].edge_index.shape[1]}')

print('\nAllocation graph construction complete.')

In [ ]:
# ─── Stage 2 training ───
train_graphs = [allocation_graphs[loc] for loc in TRAIN_LOCATIONS if loc in allocation_graphs]
test_graphs = [allocation_graphs[loc] for loc in TEST_LOCATIONS if loc in allocation_graphs]

train_dl = DataLoader(train_graphs, batch_size=1, shuffle=True)
test_dl = DataLoader(test_graphs, batch_size=1) if test_graphs else None

alloc_config = AllocationConfig(
    k_nearest_targets=K_NEAREST_TARGETS,
    hidden_dim=64,
    embedding_dim=32,
    num_layers=2,
    conv_type='hgt',
    allocation_temperature=0.1,
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=200,
    cosine_epochs=160,
    warmup_epochs=20,
    cosine_eta_min=1e-5,
    save_path=str(ALLOC_DIR / f'{EXPERIMENT_NAME}.pth'),
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

alloc_solver = AllocationSolver(alloc_config)
alloc_solver.train(
    train_dl, test_dl,
    objective_weights={
        'allocation_distance': 1.0,
        'allocation_feature_homogeneity': 0.5,
    },
    eval_every=5,
)

print(f'\nModel saved to: {alloc_config.save_path}')

In [ ]:
# ─── Inference: W_at → allocate demand to targets ───
results = {}

for loc in STUDY_REGIONS:
    if loc not in allocation_graphs:
        continue

    result = alloc_solver.predict(allocation_graphs[loc])

    if isinstance(result, tuple):
        w_at_df, demand_df = result
    else:
        w_at_df = result
        demand_df = None

    results[loc] = {
        'w_at': w_at_df,
        'demand': demand_df,
    }

    if demand_df is not None:
        print(f'{loc}: predicted demand sum={demand_df["predicted_demand"].sum():.1f} MVA')
        if 'actual_demand' in demand_df.columns:
            print(f'       actual demand sum={demand_df["actual_demand"].sum():.1f} MVA')

print('\nInference complete.')

In [ ]:
# ─── Evaluation: RMSE / MAE / Correlation ───
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error

eval_rows = []

for loc in STUDY_REGIONS:
    if loc not in results or results[loc]['demand'] is None:
        continue

    demand_df = results[loc]['demand']
    if 'actual_demand' not in demand_df.columns:
        continue

    pred = demand_df['predicted_demand'].values
    actual = demand_df['actual_demand'].values

    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)

    # Correlation
    if len(actual) > 2 and np.std(actual) > 0 and np.std(pred) > 0:
        corr, p_val = pearsonr(actual, pred)
    else:
        corr, p_val = float('nan'), float('nan')

    split = 'TRAIN' if loc in TRAIN_LOCATIONS else 'TEST'
    eval_rows.append({
        'Region': loc,
        'Split': split,
        'N_targets': len(actual),
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'Correlation': round(corr, 4),
        'p_value': round(p_val, 6),
        'Demand_sum_actual': round(actual.sum(), 1),
        'Demand_sum_pred': round(pred.sum(), 1),
    })

eval_table = pd.DataFrame(eval_rows)
print('\n=== Stage 2 Allocation Evaluation ===')
eval_table